In [ ]:
import pysam
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt

def calc_roc_auc(vcf_path, delta_scores_path, label="Benchmark"):
    # Step 1: Extract labels from the VCF file
    vcf_file = vcf_path
    vcf_reader = pysam.VariantFile(vcf_file)
    
    labels = []
    for record in vcf_reader:
        # Assuming the label is in the INFO field as a float
        label_value = float(record.info.get("INFO"))  # Replace "INFO" with the actual key
        labels.append(label_value)
    
    # Step 2: Load delta scores from CSV
    delta_scores_df = pd.read_csv(delta_scores_path)
    
    # Add labels to the DataFrame
    delta_scores_df["label"] = labels[:len(delta_scores_df)]  # Ensure lengths match
    
    # Step 3: Compute ROC curve and AUC
    delta_scores = delta_scores_df["delta_score"]
    labels = delta_scores_df["label"]

    print(f"{label} ROC-AUC Score: {roc_auc_score(labels, delta_scores)}")
    
    fpr, tpr, _ = roc_curve(labels, delta_scores)
    roc_auc = auc(fpr, tpr)
    
    # Plot the ROC curve
    plt.plot(fpr, tpr, lw=2, label=f'{label} ROC curve (area = {roc_auc:.2f})')
    
    return fpr, tpr, roc_auc

# Plot ROC curves for coding and noncoding variants
plt.figure()

# Coding variants
fpr_coding, tpr_coding, roc_auc_coding = calc_roc_auc(
    "./benchmark/ClinVar_Coding_SNV_PB.vcf", 
    "./hg38_dataset/DeltaScores_coding.csv", 
    label="Coding Variants"
)

# Noncoding variants
fpr_noncoding, tpr_noncoding, roc_auc_noncoding = calc_roc_auc(
    "./benchmark/ClinVar_NonCoding_SNV_PB.vcf", 
    "./hg38_dataset/DeltaScores_noncoding.csv", 
    label="Noncoding Variants"
)

# Plot the diagonal line (random classifier)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

# Set plot properties
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")

# Show the plot
plt.show()